# Leaf localisation — local training pipeline

Local equivalent of `notebooks/train_colab.ipynb`: downloads PlantDoc, merges in the
SoyCotton dataset, removes cross-split duplicates, then trains YOLO26m (and optionally
YOLO26s for comparison) — all on this machine's own GPU, no Drive/Colab dependency.

Why SoyCotton gets merged in: PlantDoc alone has 0 cotton images and only 33 soy images
out of 2205, and ~1% of its images leak across train/val/test (same photo saved under two
different names, landing in different splits). See `scripts/add_soycotton.py` and
`scripts/dedupe_splits.py` for the full explanation.


## 1. Locate the repo root


In [ ]:
import os
from pathlib import Path


def find_repo_root():
    # Search upward from every plausible starting point until data.yaml
    # turns up, so this works no matter where the notebook file actually
    # lives in the repo (root, notebooks/, or anywhere else).
    start_points = []

    # VS Code's Jupyter extension injects this global with the notebook's own
    # path, which works even when the kernel's cwd is unrelated (e.g. a
    # remote/JupyterHub kernel defaulting to your home directory).
    ipynb_path = globals().get("__vsc_ipynb_file__")
    if ipynb_path:
        start_points.append(Path(ipynb_path).resolve().parent)

    start_points.append(Path.cwd())

    for start in start_points:
        for candidate in [start, *start.parents]:
            if (candidate / "data.yaml").exists():
                return candidate

    return None


REPO_ROOT = find_repo_root()
assert REPO_ROOT is not None, (
    "Could not find data.yaml near the notebook's own location or the "
    f"kernel's working directory ({Path.cwd()}). Set REPO_ROOT manually to "
    "the leaf-localisation repo path, e.g.:\n"
    '  REPO_ROOT = Path("/path/to/leaf-localisation")'
)
os.chdir(REPO_ROOT)
print("Working directory set to:", Path.cwd())

## 2. Install packages


In [ ]:
%pip install -q ultralytics "numpy<2" kaggle imagehash


## 3. Check for a GPU


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"CUDA available — training will use GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU detected — training will fall back to CPU (slow). "
          "See the prerequisites note above.")

## 4. Kaggle credentials


In [ ]:
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"

if kaggle_json.exists():
    print(f"Found Kaggle credentials at {kaggle_json}")
else:
    print(
        "Kaggle credentials not found.\n\n"
        "1. Go to https://www.kaggle.com/settings -> API -> Create New Token "
        "(downloads kaggle.json)\n"
        f"2. Move it to: {kaggle_json}\n"
        "3. On macOS/Linux, also run: chmod 600 ~/.kaggle/kaggle.json"
    )

## 5. Download the PlantDoc dataset


In [ ]:
!kaggle datasets download -d nirmalsankalana/plantdoc-yolo-26-dataset -p data/raw --unzip

## 6. Inspect the raw folder layout


In [ ]:
raw_dir = Path("data/raw")
for p in sorted(raw_dir.rglob("*")):
    if p.is_dir() and len(p.relative_to(raw_dir).parts) <= 2:
        print(p)

In [ ]:
# Adjust this if the listing above shows train/val/test nested elsewhere
RAW_ROOT = Path("data/raw/yolo_dataset")

## 7. Relabel PlantDoc: collapse all 29 classes into one `leaf` class (id 0)


In [ ]:
import shutil
import sys

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from relabel import relabel_file

PROCESSED = Path("data/processed")
SPLITS = ["train", "val", "test"]

for split in SPLITS:
    src_images = RAW_ROOT / split / "images"
    src_labels = RAW_ROOT / split / "labels"
    dst_images = PROCESSED / split / "images"
    dst_labels = PROCESSED / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    n_img = 0
    for src in src_images.iterdir():
        dst = dst_images / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
        n_img += 1

    n_lbl = 0
    for src in src_labels.iterdir():
        relabel_file(src, dst_labels / src.name)
        n_lbl += 1

    print(f"{split}: {n_img} images copied, {n_lbl} labels relabeled")

## 8. Merge in the SoyCotton dataset

Downloads SoyCotton (Kellermann et al., Scientific Data 2026 — 640 field images,
7,221 soy + 5,190 cotton leaves) straight from figshare and merges it into
`data/processed`, collapsing both its classes to `leaf` (id 0) same as PlantDoc above.


In [ ]:
%run scripts/add_soycotton.py


## 9. Remove cross-split duplicates

Perceptual-hashes every image and flags near-identical photos that ended up in two
different splits (PlantDoc's Kaggle re-split reused generic filenames like `0.jpg` for
genuinely different photos, so a filename-based check isn't enough). Dry run first, then
apply — review what would be dropped before deleting anything.


In [ ]:
%run scripts/dedupe_splits.py


In [ ]:
%run scripts/dedupe_splits.py --apply


## 10. Sanity check

Expect to see (after the SoyCotton merge + dedupe above): train~1969, val~567,
test~284 images, ~19911 boxes total — matching what's been verified before training.
If these numbers are off, stop and check the steps above before training.


In [ ]:
%run scripts/verify.py

In [ ]:
from IPython.display import Image as IPImage, display

display(IPImage(filename="data/processed/sample_check.png", width=600))

## 11. Train YOLO26m

Resume-aware: if a previous run's `last.pt` exists under `RUN_NAME` below, this picks up
training from there instead of starting over — safe to re-run this cell after an
interrupted session. `batch=6` is fixed rather than autobatch (`batch=-1`): autobatch
profiles memory at *2x* imgsz to size for multi-scale training, which can OOM before
training even starts at `imgsz=1280`. Raise it if `nvidia-smi` shows headroom during
training, drop it if you see OOM warnings.


In [ ]:
import os
from pathlib import Path

from ultralytics import YOLO

MODEL = "yolo26m.pt"
EPOCHS = 120
IMGSZ = 1280
BATCH = 6
DEVICE = 0 if torch.cuda.is_available() else "cpu"
PROJECT = "runs/detect"
RUN_NAME = "train_m"

last_ckpt = Path(PROJECT) / RUN_NAME / "weights" / "last.pt"

if last_ckpt.exists():
    print(f"Found a previous run, resuming from {last_ckpt}")
    model = YOLO(str(last_ckpt))
    model.train(resume=True)
else:
    print(f"Starting a fresh run: model={MODEL} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} device={DEVICE}")
    model = YOLO(MODEL)
    model.train(
        data="data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        patience=25, save_period=10, copy_paste=0.3, mixup=0.15, seed=42,
        project=PROJECT, name=RUN_NAME,
    )


## 12. Validate


In [ ]:
from ultralytics import YOLO

WEIGHTS = f"{PROJECT}/{RUN_NAME}/weights/best.pt"

model = YOLO(WEIGHTS)
metrics = model.val(data="data.yaml", plots=True)

print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")
print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")

for conf_val in [0.25, 0.35, 0.45, 0.55, 0.58, 0.60, 0.62, 0.65]:
    m = model.val(data="data.yaml", conf=conf_val)
    print(f"conf={conf_val}: P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}")


## 13. Optional: train YOLO26s for comparison

Once `m`'s precision/recall above looks good, it's worth training `s` on the exact same
data to see how much you're actually giving up in exchange for a faster/lighter model.
Flip `RUN_TRAINING_S` to `True` when you're ready — left `False` by default so re-running
this notebook top-to-bottom doesn't kick off a second training run unintentionally.


In [ ]:
RUN_TRAINING_S = False

if RUN_TRAINING_S:
    model_s = YOLO("yolo26s.pt")
    model_s.train(
        data="data.yaml", epochs=120, imgsz=1280, batch=6, device=DEVICE,
        patience=25, save_period=10, copy_paste=0.3, mixup=0.15, seed=42,
        project="runs/detect", name="train_s",
    )
    metrics_s = model_s.val(data="data.yaml")

    print("YOLO26s:")
    print(f"Precision  : {metrics_s.box.mp:.3f}")
    print(f"Recall     : {metrics_s.box.mr:.3f}")
    print(f"mAP50      : {metrics_s.box.map50:.3f}")
    print(f"mAP50-95   : {metrics_s.box.map:.3f}")

    print("\nYOLO26m (from above):")
    print(f"Precision  : {metrics.box.mp:.3f}")
    print(f"Recall     : {metrics.box.mr:.3f}")
    print(f"mAP50      : {metrics.box.map50:.3f}")
    print(f"mAP50-95   : {metrics.box.map:.3f}")
else:
    print("RUN_TRAINING_S is False — skipping the YOLO26s comparison run.")


## 14. Label / box-area audit


In [ ]:
from pathlib import Path

SPLITS = ["train", "val", "test"]

results = []
for split in SPLITS:
    img_dir = Path(f"data/processed/{split}/images")
    lbl_dir = Path(f"data/processed/{split}/labels")
    for img_path in sorted(img_dir.glob("*")):
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue
        lines = lbl_path.read_text().strip().splitlines()
        if not lines:
            continue
        areas = [float(line.split()[3]) * float(line.split()[4]) for line in lines]
        avg_area_pct = (sum(areas) / len(areas)) * 100
        results.append((split, img_path.name, len(lines), avg_area_pct))

# Busiest images (most boxes) are your prime suspects for inconsistent labeling
results.sort(key=lambda r: r[2], reverse=True)

print(f"{'split':<6} {'image':<45} {'boxes':>6} {'avg area %':>10}")
for split, name, n_boxes, avg_area in results[:40]:
    print(f"{split:<6} {name:<45} {n_boxes:>6} {avg_area:>10.2f}")

In [ ]:
import cv2
from pathlib import Path
from PIL import Image

def show_boxes(split, stem):
    img_dir = Path(f"data/processed/{split}/images")
    lbl_dir = Path(f"data/processed/{split}/labels")

    matches = list(img_dir.glob(f"{stem}.*"))
    if not matches:
        print(f"No image file found for: {stem} in {split}")
        return
    img_path = matches[0]
    lbl_path = lbl_dir / f"{stem}.txt"
    if not lbl_path.exists():
        print(f"No label file found for: {stem} in {split}")
        return

    img = cv2.imread(str(img_path))
    if img is None:
        print(f"cv2 could not read: {img_path}")
        return

    h, w = img.shape[:2]
    lines = lbl_path.read_text().strip().splitlines()
    for line in lines:
        cls, xc, yc, bw, bh = map(float, line.split())
        x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
        x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    print(f"{split}/{stem}: {len(lines)} boxes")
    display(Image.fromarray(img[..., ::-1]))

In [ ]:
show_boxes("train", "train_tomato-yellow-leaf-c")
show_boxes("val", "train_tomato_yellow_leaf_c")

## 15. Try it on a sample image

Pick whichever weights you decided to keep (`model` = m, or `model_s` = s, if you ran
step 13). No upload widget needed locally — just point `SAMPLE_IMAGE` at any image on
disk (defaults to the first test image).


In [ ]:
CHOSEN_MODEL = model  # or model_s, if you trained it in step 13
CONF = 0.25  # lower this (e.g. 0.1) to see more/weaker boxes, raise it to see only confident ones

test_images = sorted((PROCESSED / "test" / "images").iterdir())
SAMPLE_IMAGE = test_images[0]

result = CHOSEN_MODEL.predict(str(SAMPLE_IMAGE), conf=CONF, verbose=False)[0]

print(f"{SAMPLE_IMAGE.name}: {len(result.boxes)} leaf(es) detected")
for i, box in enumerate(result.boxes):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf = float(box.conf[0])
    area_px = (x2 - x1) * (y2 - y1)
    print(f"  #{i}: conf={conf:.2f}  box=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})  area_px={area_px:.0f}")

from PIL import Image
display(Image.fromarray(result.plot()[..., ::-1]))
